# YouTube Transcriber

 O QUE FAZ:
 1. Baixa áudio de vídeos YouTube
 2. Transcreve em tempo real usando Whisper (IA da OpenAI)
 3. Armazena em banco de dados SQLite
 4. Suporta retry automático com backoff exponencial

REQUISITOS:
- yt-dlp: Download de vídeos
- openai-whisper: Transcrição de áudio
- ffmpeg: Conversão de áudio
- Outros pacotes de análise de dados


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
YouTube Transcriber
"""

import os
import sqlite3           # Banco de dados local
import logging          # Registrar eventos e erros
from datetime import datetime  # Para timestamps
from tqdm import tqdm   # Barra de progresso
import time            # Para pausas entre requisições
import random          # Para delays aleatórios
import pandas as pd    # Manipulação de dados
import matplotlib.pyplot as plt  # Gráficos
import seaborn as sns  # Estilo de gráficos

# ===== CONFIGURAÇÕES =====
DRIVE_PATH = '/content/drive/My Drive/youtube_transcriber'
DB_PATH = f'{DRIVE_PATH}/transcricoes.db'
URLS_FILE = f'{DRIVE_PATH}/urls.txt'
COOKIES_PATH = f'{DRIVE_PATH}/cookies.txt'  # cookies extraídos da minha página do youtube
WHISPER_MODEL = 'base'
BATCH_SIZE = 100
MAX_TENTATIVAS = 3  # Número de tentativas por vídeo
DELAY_MIN = 5  # Segundos mínimos entre downloads
DELAY_MAX = 15  # Segundos máximos entre downloads


In [ ]:
!pip install -q -U yt-dlp openai-whisper
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 23.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.1 MB/s eta 0:00:00


In [ ]:
# ===== SETUP LOGGING =====
log_file = f'{DRIVE_PATH}/transcriber.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


In [ ]:
# ===== FUNÇÕES =====
def criar_banco():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS transcricoes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        video_id TEXT UNIQUE NOT NULL,
        url TEXT NOT NULL,
        titulo TEXT,
        duracao_segundos INTEGER,
        transcricao TEXT,
        idioma TEXT,
        status TEXT,
        data_processamento TIMESTAMP,
        erro_mensagem TEXT,
        tentativas INTEGER DEFAULT 0
    )
    """)

    cursor.execute("PRAGMA table_info(transcricoes)")
    colunas = [c[1] for c in cursor.fetchall()]

    if "tentativas" not in colunas:
        cursor.execute(
            "ALTER TABLE transcricoes ADD COLUMN tentativas INTEGER DEFAULT 0"
        )

    conn.commit()
    conn.close()
    logger.info("✅ Banco de dados pronto")

def carregar_urls():
    """Carrega URLs do arquivo"""
    with open(URLS_FILE, 'r', encoding='utf-8') as f:
        urls = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    logger.info(f"✅ Carregadas {len(urls)} URLs")
    return urls

def extrair_video_id(url):
    """Extrai o ID do vídeo da URL"""
    try:
        if "youtube.com/watch?v=" in url:
            return url.split("v=")[1].split("&")[0]
        elif "youtu.be/" in url:
            return url.split("youtu.be/")[1].split("?")[0]
    except:
        pass
    return None

def get_nao_processados(urls):
    """Retorna apenas as URLs que ainda não foram processadas"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT video_id FROM transcricoes")
    processados = {row[0] for row in cursor.fetchall()}
    conn.close()

    nao_processados = []
    for url in urls:
        vid = extrair_video_id(url)
        if vid and vid not in processados:
            nao_processados.append(url)
    return nao_processados

def baixar_audio_com_retry(url, video_id):
    """
    Baixa áudio com retry automático e backoff exponencial
    Suporta cookies do YouTube se disponível
    """
    import yt_dlp

    usar_cookies = os.path.exists(COOKIES_PATH)

    for tentativa in range(MAX_TENTATIVAS):
        try:
            # Delay aleatório ANTES da tentativa (se não é a primeira)
            if tentativa > 0:
                tempo_espera = (2 ** tentativa) * 10  # 10s, 20s, 40s
                logger.info(f"  ⏳ Tentativa {tentativa+1}/{MAX_TENTATIVAS}. Aguardando {tempo_espera}s...")
                time.sleep(tempo_espera)
            else:
                # Delay aleatório na primeira tentativa também
                delay = random.uniform(DELAY_MIN, DELAY_MAX)
                time.sleep(delay)

            ydl_opts = {
                'format': 'bestaudio/best',
                'postprocessors': [{
                    'key': 'FFmpegExtractAudio',
                    'preferredcodec': 'mp3',
                    'preferredquality': '128',
                }],
                'outtmpl': f'/tmp/{video_id}',
                'socket_timeout': 30,
                'retries': 3,
                'quiet': True,
                'no_warnings': True,
                'http_headers': {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
                }
            }

            # Adiciona cookies se disponível
            if usar_cookies:
                ydl_opts['cookiefile'] = COOKIES_PATH
                logger.info(f"  🔐 Usando cookies para autenticação")

            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                duracao = info.get('duration', 0)
                titulo = info.get('title', 'Sem título')

            # Delay após sucesso para não sobrecarregar
            time.sleep(random.uniform(2, 5))

            logger.info(f"  ✅ Download bem-sucedido")
            return f'/tmp/{video_id}.mp3', duracao, titulo

        except Exception as e:
            erro_msg = str(e)[:100]

            if tentativa == MAX_TENTATIVAS - 1:
                logger.warning(f"  ❌ Falha final após {MAX_TENTATIVAS} tentativas: {erro_msg}")

                # Tenta sugerir solução baseada no erro
                if "Sign in to confirm" in str(e):
                    logger.warning(f"  💡 Dica: Use cookies do YouTube ou aumente DELAY_MAX")
                elif "Video unavailable" in str(e):
                    logger.warning(f"  💡 Dica: Vídeo pode estar privado ou deletado")

                return None, None, None

def transcrever_audio(caminho):
    """Transcreve áudio usando Whisper"""
    try:
        import whisper
        model = whisper.load_model(WHISPER_MODEL)
        resultado = model.transcribe(caminho, language="pt", verbose=False, fp16=True)
        return resultado.get('text', ''), resultado.get('language', 'pt')
    except Exception as e:
        logger.warning(f"⚠️  Erro ao transcrever: {str(e)[:100]}")
        return None, None

def salvar_no_banco(video_id, url, titulo, duracao, transcricao, idioma, status, erro=None, tentativas=0):
    """Salva ou atualiza registro no banco"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT OR REPLACE INTO transcricoes
        (video_id, url, titulo, duracao_segundos, transcricao, idioma, status, data_processamento, erro_mensagem, tentativas)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (video_id, url, titulo, duracao, transcricao, idioma, status, datetime.now(), erro, tentativas))
    conn.commit()
    conn.close()

def limpar_audio(caminho):
    """Remove arquivo de áudio temporário"""
    try:
        if os.path.exists(caminho):
            os.remove(caminho)
    except:
        pass

def processar_videos(urls_restantes):
    """Loop principal de processamento"""
    print("\n" + "="*60)
    print("🎬 INICIANDO PROCESSAMENTO")
    print("="*60 + "\n")

    processados = 0
    erros = 0
    tempo_inicio = time.time()

    for idx, url in enumerate(urls_restantes, 1):
        try:
            video_id = extrair_video_id(url)
            if not video_id:
                continue

            print(f"[{idx}/{len(urls_restantes)}] {video_id[:8]}... ", end="", flush=True)

            # Baixa com retry
            audio, duracao, titulo = baixar_audio_com_retry(url, video_id)
            if not audio:
                salvar_no_banco(video_id, url, None, None, None, None, "erro_download",
                              tentativas=MAX_TENTATIVAS)
                print("❌ Download")
                erros += 1
                continue

            # Transcreve
            transcricao, idioma = transcrever_audio(audio)
            limpar_audio(audio)

            if transcricao:
                salvar_no_banco(video_id, url, titulo, duracao, transcricao, idioma, "sucesso")
                print("✅")
                processados += 1
            else:
                salvar_no_banco(video_id, url, titulo, duracao, None, None, "erro_transcricao")
                print("❌ Transcricao")
                erros += 1

        except KeyboardInterrupt:
            print("\n\n⛔ Parado pelo usuário")
            break
        except Exception as e:
            print(f"❌ {str(e)[:30]}")
            erros += 1

    tempo_total = (time.time() - tempo_inicio) / 3600

    print("\n" + "="*60)
    print("ESTATÍSTICAS FINAIS")
    print("="*60)
    print(f"✅ Sucesso: {processados}")
    print(f"❌ Erros: {erros}")
    print(f"⏱️  Duração total: {tempo_total:.1f} horas")
    print(f"📁 Banco de dados: {DB_PATH}")
    print(f"📋 Log: {log_file}")
    print("="*60 + "\n")

# ===== MAIN =====

if __name__ == "__main__":
    print("\n🔧 Configurações Ativas:")
    print(f"  • MAX_TENTATIVAS: {MAX_TENTATIVAS}")
    print(f"  • DELAY: {DELAY_MIN}-{DELAY_MAX}s aleatório")
    print(f"  • Cookies: {'Sim' if os.path.exists(COOKIES_PATH) else 'Não (opcional)'}")
    print(f"  • Model Whisper: {WHISPER_MODEL}")

    criar_banco()
    todas_urls = carregar_urls()
    urls_restantes = get_nao_processados(todas_urls)

    if not urls_restantes:
        print("\n✅ Todos os vídeos já foram processados!")
    else:
        print(f"\n📊 Processando {len(urls_restantes)} vídeos...\n")
        processar_videos(urls_restantes)


🔧 Configurações Ativas:
  • MAX_TENTATIVAS: 3
  • DELAY: 5-15s aleatório
  • Cookies: Sim
  • Model Whisper: base

✅ Todos os vídeos já foram processados!


In [ ]:

db_path = '/content/drive/My Drive/youtube_transcriber/transcricoes.db'

conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM transcricoes", conn)

print("="*80)
print("📊 ANÁLISE DAS 289 TRANSCRIÇÕES")
print("="*80)

print(f"\n✅ Total: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"\n📋 Colunas:")
for col in df.columns:
    dtype = df[col].dtype
    non_null = df[col].notna().sum()
    print(f"  • {col} ({dtype}) - {non_null} não-nulos")

conn.close()

📊 ANÁLISE DAS 289 TRANSCRIÇÕES

✅ Total: 539 linhas x 11 colunas

📋 Colunas:
  • id (int64) - 539 não-nulos
  • video_id (object) - 539 não-nulos
  • url (object) - 539 não-nulos
  • titulo (object) - 429 não-nulos
  • duracao_segundos (float64) - 429 não-nulos
  • transcricao (object) - 426 não-nulos
  • idioma (object) - 426 não-nulos
  • status (object) - 539 não-nulos
  • data_processamento (object) - 539 não-nulos
  • erro_mensagem (object) - 0 não-nulos
  • tentativas (int64) - 539 não-nulos


In [ ]:
# CSV
df.to_csv('transcricoes_completo.csv', index=False)
print("✅ Salvo: transcricoes_completo.csv")

# Baixar
from google.colab import files
files.download('transcricoes_completo.csv')

✅ Salvo: transcricoes_completo.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>